Let’s build the first LightGBM/TFT model for forecasting.

So let’s build a proper plan, start simple with LightGBM first, and then move to TFT afterward.

This way, you'll have:

1. A strong LightGBM baseline first
2. Then a powerful Generative Model (TFT) second

1. Plan for Forecasting Model Development

    First Model: LightGBM Baseline

      Why first LightGBM?

      Easy to train, Interpretable (feature importance),Great tabular forecaster to benchmark

      Goal: Build a basic sales regression model. Predict Sales for each Store on a given Date using features we engineered.

      Evaluate using RMSE (Root Mean Squared Error).



2. Data Preparation for LightGBM

      From rossmann_store_sales_master_train:

      Target -> Sales
      Numerical Inputs -> Open, Promo, SchoolHoliday, IsWeekend, CompetitionDistance, MonthsSinceCompetitionOpen
      Categorical Inputs -> Store, StoreType, Assortment, DayOfWeek, Month, Year, StateHoliday, Promo2, IsPromo2Active

      Notes:

      Categorical features must be label encoded for LightGBM.

      Sales should be predicted only for rows where Open = 1 (store is open).

      Log1p transform on Sales helps because sales are right-skewed.

3. Step-by-Step Modeling Plan

      Step	Task
      1	Load train/test master tables.

      2	Filter only Open=1 records

      3	Feature selection (numerical + categorical)

      4	Encode categorical features

      5	Train LightGBM model

      6	Evaluate on a validation split
      
      7	Predict sales on test data

4. Preprocessing and Feature Handling
Some preprocessing we'll do:

      Fill missing values in competition-related features.

      Encode categorical variables using Label Encoding.

      Optionally apply log1p transformation on Sales during training (then inverse at prediction time).

5. Evaluation Metrics

      Metric	Description

      RMSE (Root Mean Squared Error)	Sensitive to large errors, great for sales data
      
      MAE (Mean Absolute Error)	Optional secondary check

6. LightGBM First Baseline Model Implementation

In [0]:
# 1. Load Libraries
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

# 2. Load Master Train Table
train = spark.table('rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_train').toPandas()

# 3. Filter Open Stores Only
train = train[train['Open'] == 1]

# 4. Select Features
target = 'Sales'
features = [
    'Promo', 'SchoolHoliday', 'IsWeekend', 'CompetitionDistance', 'MonthsSinceCompetitionOpen',
    'StoreType', 'Assortment', 'DayOfWeek', 'Month', 'Year', 'StateHoliday', 'Promo2', 'IsPromo2Active'
]

X = train[features]
y = np.log1p(train[target])  # log1p transformation to handle skewness

# 5. Label Encoding for Categorical Columns
cat_features = ['StoreType', 'Assortment', 'DayOfWeek', 'Month', 'Year', 'StateHoliday', 'Promo2', 'IsPromo2Active']
for col in cat_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# 6. Train-Validation Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. LightGBM Dataset
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_features)

# 8. LightGBM Parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

# 9. Train Model (corrected)
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    num_boost_round=500,   # <--- hard limit
    callbacks=[
        lgb.early_stopping(stopping_rounds=10),
        lgb.log_evaluation(period=10)
    ]
)


# 10. Evaluate
y_pred = model.predict(X_val)
rmse = mean_squared_error(np.expm1(y_val), np.expm1(y_pred), squared=False)
print(f"Validation RMSE: {rmse:.2f}")


First: What is RMSE?
RMSE = Root Mean Squared Error

Mathematically:

RMSE = sqrt( (1/N) * sum( (predicted - actual)^2 ) )

It measures the average difference between your predicted sales and the actual sales.

RMSE penalizes large errors more than small ones (because of squaring).

Lower RMSE = better model (closer predictions).


Plot Feature Importance for your LightGBM Model

This will show which features (e.g., Promo, CompetitionDistance, Holiday, Month)
are driving your Rossmann store sales predictions the most.



In [0]:
# 1. Import Plotting Library
import matplotlib.pyplot as plt
import lightgbm as lgb

# 2. Create Feature Importance
importance = model.feature_importance()
feature_names = model.feature_name()

# 3. Organize into a DataFrame for better plotting
import pandas as pd

fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

# 4. Plot the Feature Importance
plt.figure(figsize=(10, 6))
plt.barh(fi_df['feature'], fi_df['importance'])
plt.xlabel('Feature Importance')
plt.title('LightGBM Feature Importance for Rossmann Sales Forecasting')
plt.gca().invert_yaxis()
plt.show()


What we have done so far

a.  Data Engineering

b.  LightGBM Model Training

c.  Feature Importance Analysis

Now, let’s predict on Test Data (rossmann_store_sales_master_test) to generate future sales forecasts!

##Step-by-Step Code for Test Data Prediction

1. Load Test Data

In [0]:
# Load test dataset
test = spark.table('rcg_store_catalog.rcg_store_schema.rossmann_store_sales_master_test').toPandas()


2. Apply Same Feature Selection (without 'Store')

In [0]:
# Use the same features as during training
test_features = [
    'Promo', 'SchoolHoliday', 'IsWeekend', 'CompetitionDistance', 'MonthsSinceCompetitionOpen',
    'StoreType', 'Assortment', 'DayOfWeek', 'Month', 'Year', 'StateHoliday', 'Promo2', 'IsPromo2Active'
]
X_test = test[test_features]


3. Apply Same Label Encoding for Categorical Features

In [0]:
# Apply label encoding for categorical features
for col in ['StoreType', 'Assortment', 'DayOfWeek', 'Month', 'Year', 'StateHoliday', 'Promo2', 'IsPromo2Active']:
    le = LabelEncoder()
    X_test[col] = le.fit_transform(X_test[col].astype(str))

4. Make Predictions

In [0]:
# Predict on Test Data
y_test_pred_log = model.predict(X_test)

# Inverse log1p to get back to real Sales numbers
y_test_pred_sales = np.expm1(y_test_pred_log)



In [0]:
# Add Predictions to Test Data
test['PredictedSales'] = y_test_pred_sales

# Optional: Round Sales to nearest integer
test['PredictedSales'] = test['PredictedSales'].round().astype(int)

# See sample output
test[['Id', 'Store', 'Date', 'PredictedSales']].head()


In [0]:
# Remember:
# y_val = validation target (log1p(Sales))
# y_pred = validation prediction (log1p)

# Undo log1p
y_val_real = np.expm1(y_val)
y_pred_real = np.expm1(y_pred)

# Create a DataFrame to compare
validation_comparison = pd.DataFrame({
    'ActualSales': y_val_real,
    'PredictedSales': y_pred_real
})

# See first few rows
validation_comparison.head(10)


In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.scatter(validation_comparison['ActualSales'], validation_comparison['PredictedSales'], alpha=0.5)
plt.plot([0, 20000], [0, 20000], color='red', linestyle='--')  # perfect prediction line
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales (Validation Set)')
plt.show()



1. Inverse Transform to Real Sales

      Because both y_val and y_pred are still in log1p space.
      We need to expm1 them to get real sales units.

In [0]:
# 1. Undo log1p transformation
y_val_real = np.expm1(y_val)      # Actual Sales
y_pred_real = np.expm1(y_pred)    # Predicted Sales

# 2. Create comparison DataFrame
import pandas as pd

validation_comparison = pd.DataFrame({
    'ActualSales': y_val_real,
    'PredictedSales': y_pred_real
})

# Optional: Calculate Error
validation_comparison['Error'] = validation_comparison['ActualSales'] - validation_comparison['PredictedSales']
validation_comparison['AbsoluteError'] = validation_comparison['Error'].abs()


In [0]:
# 3. See top records
validation_comparison.head(10)

In [0]:
# 4. Quick error metrics
mean_absolute_error = validation_comparison['AbsoluteError'].mean()
print(f"Mean Absolute Error on Validation Set: {mean_absolute_error:.2f}")

In [0]:
# 5. Plot scatter
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.scatter(validation_comparison['ActualSales'], validation_comparison['PredictedSales'], alpha=0.5)
plt.plot([0, validation_comparison['ActualSales'].max()], [0, validation_comparison['ActualSales'].max()], color='red', linestyle='--')  # Perfect prediction line
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales (Validation Set)')
plt.grid(True)
plt.show()



Generative Forecasting using Temporal Fusion Transformer (TFT)

LightGBM	vs  TFT

Good for point forecasts -	Good for generating multiple possible futures

Doesn't capture time sequence fully	- Fully time-series aware

No direct probabilistic forecasting	- Gives prediction intervals naturally

Cannot simulate scenarios (what-if)	- Can simulate sales under different promos, events

TFT will make your forecasting real-world ready:
– Flexible, dynamic, generative, risk-aware.

What We Will Do in the Next Phase

Prepare the input data in time-series structure for TFT

Define static, past observed, and known future features

Train a Temporal Fusion Transformer (TFT) model

Generate multiple future sales scenarios

(Later) Integrate RAG + Agent layers for dynamic context retrieval if needed

TFT expects special structure, different from LightGBM flat tables:


Feature Type	- Description	- Example

Static Features	- Store properties	- StoreType, Assortment

Observed Past Features	- Features you observe over time	- Sales, Promo last month

Known Future Features -	Calendar/promo events known beforehand	- Future Holidays, Future Promo

Target	- What you are forecasting	- Sales

1. Define Feature Groups
From your master train/test tables:


Category	- Features

Static (by store)	- StoreType, Assortment

Past Observed	- Past Sales, Past Open Flag, Past Promo Flag

Known Future Inputs -	Future Promo flag, Holiday flag, SchoolHoliday flag

Time Features -	Date, Month, DayOfWeek, WeekOfYear

Now we are moving from tabular prediction to sequence-based forecasting
where model sees sales patterns day-by-day, week-by-week, store-by-store."

You are truly entering Generative Forecasting Phase! 